# 習題：數學物件、Python 型別與 AI 中的數值表示

在數學中，我們經常接觸下列概念：

- 自然數（Natural Numbers）
- 整數（Integers）
- 有理數（Rational Numbers）
- 實數（Real Numbers）
- 複數（Complex Numbers）
- 向量（Vectors）
- 矩陣（Matrices）
- 集合（Sets）
- 命題的真假（Truth Values）

本習題將探討：

```text
數學物件
    ↓
Python 型別
    ↓
CPython 實作
    ↓
AI 中的數值表示
```

---

## 第一部分：Python 如何表示數學物件？

查閱 Python 文件或自行測試，完成下表。

| 數學概念 | Python 型別 | 範例 |
|----------|------------|------|
| 自然數 | int | 5 |
| 整數 | int | -5 |
| 有理數 | fractions.Fraction | 1/2 |
| 實數 | float | 1.23 |
| 複數 | complex | 3+3j |
| 向量 | list | [1, 2, 3] |
| 矩陣 | list | [[1, 2], [3, 4]]|
| 集合 | set | {1,2} |
| 命題真假 | bool | True |

並回答下列問題：

1. 哪些是 Python 的內建型別（Built-in Types）？int、float、complex、list、set、bool
2. 哪些需要額外模組才能較完整地表示？有理數、向量、矩陣
3. Python 的型別分類與數學中的分類是否完全一致？不完全一致

---

## 第二部分：數學概念與 Python 型別

數學上有下列包含關係：

$$
\mathbb N
\subseteq
\mathbb Z
\subseteq
\mathbb Q
\subseteq
\mathbb R
\subseteq
\mathbb C
$$

請回答：

1. Python 是否為上述每一個數系都提供獨立型別？否
2. 為什麼自然數與整數都使用 `int` 表示？因為自然數只是整數的子集，在電腦底層的儲存與運算方式完全相同，統一使用 int 能避免相減變負數時產生繁瑣的型別轉換。
3. Python 為什麼提供 `complex` 型別？為了讓科學與工程領域能以極簡語法（如 3 + 4j）直接進行複數運算，並在底層享有 C 語言等級的高效能計算能力。
4. Python 為什麼沒有內建的有理數型別？因為浮點數（float）已能滿足絕大多數日常與科學計算的效能及精度需求
5. 下列三個物件的型別有何不同？int float complex

```python
1
1.0
1+0j
```

---

## 第三部分：CPython 如何實作這些型別？

除了知道 Python 使用哪些型別表示數學物件之外，

我們也希望了解：

> Python 物件在 CPython 中是如何實作的？

請查閱資料，完成下表。

| 數學概念 | Python 型別 | CPython 實作 | 主要特點 |
|----------|------------|-------------|----------|
| 自然數 | `int` | PyLongObject | 底層使用「動態長度陣列」儲存 30 位元（或 15 位元）的數字塊（digits），無記憶體位元上限（不會溢位） |
| 整數 | `int` | PyLongObject | 採用任意精度整數，位元長度隨數值大小動態擴充；CPython 內部會預先快取 -5 到 256 的小整數物件以優化效能 |
| 實數 | `float` | PyFloatObject | 底層直接封裝 C 語言的 double（雙精度浮點數），佔用 64 位元（1 位正負號、11 位指數、52 位尾數），符合 IEEE 754 標準，存在二進位進位誤差。 |
| 複數 | `complex` | PyComplexObject | 結構體內部直接包含兩個 C 語言的 double 變數（分別代表 real 實部與 imag 虛部），共佔用 128 位元記憶體空間。 |
| 集合 | `set` | PySetObject | 基於雜湊表實作，元素必須是可雜湊（Hashable/Immutable）物件；搜尋、新增與刪除的平均時間複雜度皆為 O(1) |
| 命題真假 | `bool` | PyBoolObject | 是 int 的 C 語言子類別；全域只存在兩個單例物件（Py_True 與 Py_False），底層數值即為 1 與 0。 |
| 串列 | `list` | PyListObject | 底層為動態指標陣列，連續儲存指向元素的指標；支援 O(1) 隨機存取（索引取值），並採用預分配擴容策略 |

---

## 第四部分：深入探討各種型別

請針對第三部分中的每一種型別回答：

1. 它是否為 Python 的內建型別？
2. 對應的 CPython 物件結構名稱為何？
3. 它如何儲存資料？
4. 是否有固定大小限制？
5. 若資料規模持續增加，如何擴充儲存空間？
自然數（int）：
1.是。
2.PyLongObject
3.採用變長數字塊陣列（Array of Digits）。在 64 位元系統下，CPython 將大整數拆解成以2的30次方為底的位元塊（digits），並存放在動態陣列中；另有標頭記錄陣列長度與正負號。
4.無固定上限（僅受限於系統可用記憶體）。
5.當數值增大超過當前陣列容量時，CPython 會重新分配更大的記憶體空間並搬移數字塊陣列（類似 realloc 機制）。
整數（int）：
1.是。
2.PyLongObject
3.與自然數完全相同，採用變長數字塊陣列儲存。CPython 啟動時會預先建立 -5 至 256 的小整數單例物件（Small Integer Cache）供全域共用以優化效能。
4.無固定上限（僅受限於系統可用記憶體）。
5.當數值增大超過當前陣列容量時，CPython 會重新分配更大的記憶體空間並搬移數字塊陣列（類似 realloc 機制）。
實數（float）:
1.是。
2.PyFloatObject
3.直接封裝 C 語言的 double 變數（符合 IEEE 754 雙精度 64 位元浮點數規格：1 位正負號、11 位指數、52 位尾數）。
4.有固定限制（固定佔用 64 位元/8 位元組，數值範圍約介於 ±2.22×10⁻³⁰⁸ 至 ±1.79×10³⁰⁸ ）。
5.無法擴充。數值若超過上限會引發 OverflowError 或轉為 inf（無窮大）；低於下限則會發生下溢（Underflow）。
複數（complex）
1.是。
2.PyComplexObject（內含 C 結構體 Py_complex）
3.結構內部包含兩個 C 語言的 double 變數，分別儲存實部（real）與虛部（imag）。
4.有固定限制（固定佔用 128 位元/16 位元組，由兩個 64 位元的 double 組成）。
5.無法擴充。當實部或虛部的數值超過雙精度浮點數極限時，處理機制與 float 相同。
集合（set）
1.是。
2.PySetObject
3.採用開址雜湊表（Open Addressing Hash Table）。內部維護 entry 陣列，儲存元素的雜湊值（hash value）與指向實際物件的指標。
4.無固定上限（元素數量僅受限於系統可用記憶體）。
5.當載入因子（Load Factor）超過2/3時，CPython 會觸發自動重雜湊（Rehash），重新分配 2 到 4 倍大的記憶體空間並重新計算所有元素的存放位置。
命題真假（bool）
1.是。
2.PyBoolObject（繼承自 PyLongObject）
3.結構本質為 int。CPython 在初始化時即建立全域唯二的單例物件 Py_True 與 Py_False，其底層數值分別儲存為 1 與 0。
4.有固定限制（值域僅包含 True 與 False 兩種狀態）。
5.無法擴充。變數指派僅為指向全域單例物件的指標切換。
串列（list）
1.是。
2.PyListObject
3.採用動態指標陣列（Array of Pointers）。內部維護一段連續記憶體，專門存放指向各個元素物件的 C 語言指標（PyObject*）。
4.無固定上限（元素數量僅受限於系統可用記憶體）。
5.採用超額分配（Over-allocation）策略。當元素增加導致容量不足時，CPython 會依公式（約增加1/8容量加上常數）重新分配更大的連續記憶體並搬移指標陣列。
---

## 第五部分：Python 整數與浮點數

執行下列程式：

```python
print(10**100)
print(10**1000)
```

回答：

1. 為什麼 Python 能表示如此大的整數？因為 Python 的整數類型採用了動態記憶體分配與大數運算機制
2. Python 的 `int` 是否具有固定的位元數限制？沒有固定的位元數限制。
3. 什麼是 Arbitrary Precision（任意精度）？指計算時數值的精度與範圍只受限於系統可用記憶體，而非受限於硬體暫存器的位元數。
4. CPython 使用什麼物件來實作 `int`？底層使用 C 語言的 PyLongObject 結構體，以動態陣列的形式分段儲存巨大整數。
5. Python 整數大小真正的限制是什麼？真正的限制在於電腦實體記憶體的容量大小，以及處理超大數字時的計算時間與 CPU 效能。

---

再執行：

```python
0.1 + 0.2
```

```python
0.1 + 0.2 == 0.3
```

```python
format(0.1, ".20f")
```

回答：

1. 為什麼結果與數學上的答案不同？因為十進位小數 0.1 與 0.2 轉為二進位時是無限循環小數，電腦受限於 64 位元記憶體長度必須進行截斷與捨入，導致微小的微量誤差，相加後誤差被累積顯現出來。
2. 浮點數是否能精確表示所有小數？不能。 浮點數只能精確表示分母為 2 的冪次（例如 1/2=0.5、1/4=0.25、1/8=0.125）的十進位小數；其餘大部分小數（如 0.1、0.2、0.3）都只能以近似值儲存。
3. 什麼是浮點數的精度（precision）？精度是指數值能維持的有效位數與準確程度。以 IEEE 754 雙精度浮點數為例，它擁有 52 位元的尾數（Significand），能提供相當於十進位約 15 到 17 位的有效數字精度。
4. 什麼是浮點數的範圍（range）？範圍是指該數據型態所能表示的最大與最小數值區間。雙精度浮點數使用 11 位元來儲存指數（Exponent），其可表示的範圍約在10⁻³⁰⁸至10³⁰⁸之間。
5. 精度與範圍有何不同？範圍指的是能代表數值的大小的極限（極大或極小），而精度指的是在這個數值範圍內能保有幾位的精確度。

---

## 第六部分：AI 模型中的數值表示

人工智慧模型需要儲存大量參數（Parameters）與權重（Weights）。

請查閱資料並回答：

1. AI 模型中的權重通常使用什麼型別儲存？BF16 或 FP16
2. 什麼是 FP64？使用 64 個位元（1 符號、11 指數、52 尾數）儲存數值，提供極高精度與極大範圍，主要用於科學計算而非 AI。
3. 什麼是 FP32？使用 32 個位元（1 符號、8 指數、23 尾數）儲存數值，是傳統深度學習訓練的標準高精度格式。
4. 什麼是 FP16？使用 16 個位元（1 符號、5 指數、10 尾數）儲存數值，可將記憶體用量減半並大幅加速計算，但數值範圍較小易溢位。
5. 什麼是 BF16？使用 16 位元（1 符號、8 指數、7 尾數），保有與 FP32 相同的數值範圍，能有效防止梯度溢位且更適合 AI 訓練。
6. 為什麼大型 AI 模型不直接使用 Python 的 `int` 作為權重？Python 的 int 是動態長度整數且無法表示小數，無法進行微小的梯度更新，且高額的記憶體與運算開銷會嚴重拖慢模型效能。

---

## 第七部分：比較不同數值表示方式

完成下表。

| 表示方式 | 位元數 | 是否有誤差 | 主要用途 |
|----------|--------|------------|----------|
| int | C/C++ 等通常為 32 或 64 bits；Python int 為變長動態位元 | 無誤差 | 計算次數、陣列索引、邏輯控制、量化模型 |
| FP64 | 64 bits | 有誤差（有限精度小數） | 高精度科學計算、物理模擬、金融精算 |
| FP32 | 32 bits | 有誤差（有限精度小數 | 傳統 AI 模型訓練與推理、標準浮點運算 |
| FP16 | 16 bits | 有誤差（精度較低，易溢位） | 混合精度 AI 訓練、模型推理加速、省記憶體 |
| BF16 | 16 bits | 有誤差（精度比 FP16 低，但範圍等同 FP32） | 現代大型 AI 模型（LLM）訓練與推理 |

並回答：

1. 哪種表示方式最精確？FP64
2. 哪種表示方式最節省記憶體？FP16 與 BF16
3. 哪種表示方式最常見於 AI 訓練？BF16
4. 哪種表示方式最常見於 AI 推論（Inference）？FP16 或 BF16

---

## 第八部分：綜合思考

### 思考題一

Python 官方文件指出：

```python
int
```

具有 Unlimited Precision（無限制精度）。

請回答：

1. Unlimited Precision 是什麼意思？指整數不受限於硬體固定位元，不會發生位元溢位，可處理任意長度的數字。
2. 它是否代表可以儲存無限大的整數？理論上可以無限大，但實務上受限於電腦實體記憶體容量與計算時間。
3. CPython 如何實現這項特性？在底層使用動態陣列將大整數拆分成多個區塊儲存，並透過自訂的大數加減與乘法演算法來進行進位與計算。

---

### 思考題二

比較：

```python
int
```

與

```python
float
```

回答：

1. 為什麼 `int` 幾乎沒有固定上限？int 是動態長度物件，能根據數字大小向作業系統申請更多記憶體（動態擴充陣列），因此上限僅取決於電腦的 RAM 容量。
2. 為什麼 `float` 卻存在範圍限制？為了配合 CPU 硬體（FPU）加速與跨平台標準，Python 的 float 底層採用固定的 IEEE 754 雙精度 64 位元 規格，其指數位元限制了數值範圍（約在10⁻³⁰⁸至10³⁰⁸之間）。
3. 為什麼 `float` 可能產生誤差？十進位的常見小數（如 0.1、0.2）轉成二進位時會變成無限循環小數，而 64 位元浮點數只能保留 52 位元的尾數精度，因此在儲存時被強制截斷四捨五入，導致微小誤差。

---

### 思考題三

比較下列兩種設計理念：

#### Python 的整數

```text
Arbitrary Precision
正確性優先
```

#### AI 模型常用的 FP16、BF16

```text
Limited Precision
效率優先
```

請說明：

1. 兩種設計理念的差異。Python 的整數為了追求結果的絕對精確而動態調整記憶體，而 AI 模型採用的 FP16/BF16 則是為了極致的運算效率與顯存節省而選擇固定低精度。
2. 各自的優點與缺點。Python 整數的優點是不會溢位且絕對精確，但缺點是運算速度慢且記憶體開銷大；FP16/BF16 的優點是運算極快且顯存佔用極低，但缺點是存在精度捨入誤差與數值溢位風險。
3. 為什麼大型語言模型通常選擇較低精度的表示方式？大型語言模型選擇低精度是因為神經網路具有高容錯性，適度降低精度能大幅節省顯存並利用硬體加速，在不影響生成品質的前提下實現海量參數的平行運算。
4. 如果所有 AI 模型都改用 FP64，可能會產生哪些問題？若全部改用 FP64，模型顯存需求與傳輸頻寬將暴增數倍，導致硬體成本與訓練推理時間成倍飆升，且高精度的代價完全無法換來相應的模型品質提升。

---

### 挑戰題

請用自己的話解釋：

```python
x = 10**1000
```

為什麼 Python 可以正常運作；

而大型 AI 模型卻通常選擇使用：

```text
FP32
FP16
BF16
```

而不是任意精度整數或高精度浮點數？

請從：

- 記憶體使用量
任意精度整數或高精度浮點數的結構為動態變動，會產生額外的管理開銷。若用於存放數百億個參數，顯示記憶體（VRAM）會瞬間爆滿。相對地，固定長度的格式能大幅壓縮體積：FP32 每個參數佔用 4 位元組（Bytes），而 FP16 與 BF16 僅佔 2 位元組。以 700 億參數（70B）的模型為例，採用 FP16/BF16 載入權重的 VRAM 需求能從 280 GB 降低至 140 GB，大幅降低硬體門檻。
- 計算速度
任意精度的算術無法直接由 CPU 或 GPU 的硬體電路單步執行，必須依賴軟體層面的迴圈與拆解，運算速度極慢。FP32、FP16 與 BF16 屬於固定長度資料型態，可以直接交由硬體內部的邏輯閘執行，具備極高的單週期計算效率。
- 數值精度
神經網路本質上是機率統計模型，對微小的數值誤差具備高度容錯性（Resilience），微小的精度損失不會影響最終的語意理解或預測結果。
相較於精度，數值範圍（Dynamic Range）更加重要。例如 BF16 雖然犧牲了部分尾數精度，但保持了與 FP32 相同的指數範圍，成功避免了訓練過程中的梯度溢位問題。
- GPU 運算效率
現代 AI 晶片專為矩陣乘法設計了專用硬體加速單元，能直接對 FP16 與 BF16 進行平行運算。其計算吞吐量（TFLOPS）通常是 FP32 的兩倍以上。此外，將資料型態從 32 位元縮減至 16 位元，能在相同的記憶體頻寬下傳輸兩倍的資料量，大幅緩解 VRAM 搬運資料的效能瓶頸。
等角度進行討論。
